# Level 2 + 3 — Baseline exam, QLoRA fine-tune, exam again (Colab Pro L4)

**Goal:** teach Qwen2.5-7B-Instruct the MBG WhatsApp-assistant behaviour from the cleaned training examples
(steps 8 + 8b): answer only from the business's memory, hand over with `[HANDOVER]` + the hand-over message when the
memory has no answer, reply in the customer's language, short. Then measure it.

## What this notebook does

1. **Baseline exam** — the untrained model answers the **257 golden questions** (businesses never seen in
   training, every reference reviewed by Claude). Scored mechanically by `src/mbg_eval/metrics.py`.
   Cached on HF after the first run.
2. **QLoRA fine-tune** — 4-bit base + LoRA adapters (r=16, all linear layers), loss on the reply only.
   Every checkpoint is pushed to a private HF model repo → a disconnect resumes from the last one.
3. **Same exam again** with the trained adapter → comparison table, examples side by side, verdict.

## MODE (Cell 4)

- **`trial`** (run this first) — 3,000 examples × 1 epoch. Does it learn the behaviour? ~35–45 min on the L4.
- **`full`** — only after the trial says READY. All ~10,000 examples × 2 epochs. ~2–3 h on the L4
  (or ~1 h on the A100; the batch size adapts automatically).

## GPU

Pick **L4** (Runtime → Change runtime type). A T4 is refused (too small/slow). Disconnect when Cell 13 is done.

## Health checks

- Cell 7 prints the exact part of an example the model learns from (must be only the reply + `<|im_end|>`).
- Training stops by itself if the loss becomes NaN / explodes; the validation loss must go down.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)
else:
    # Already cloned in this runtime -> pull the latest code (fixes pushed since the last run).
    r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass("GitHub token (to pull updates): ")
        r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only",
                            REPO_URL.replace("https://", f"https://{tok}@"), "main"], capture_output=True, text=True)
    print("git pull:", (r.stdout or r.stderr).strip()[-300:])

print("code version:", subprocess.run(["git", "-C", REPO_PATH, "log", "-1", "--oneline"],
                                      capture_output=True, text=True).stdout.strip())
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

# transformers: keep Colab's version if it is new enough; peft / bitsandbytes / accelerate: always the latest,
# so they match whatever transformers Colab ships (4.x or 5.x — src/train/qlora.py handles both).
for _args in (["huggingface_hub>=0.24", "transformers>=4.45", "safetensors"],
              ["-U", "peft", "bitsandbytes", "accelerate"]):
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_args], capture_output=True, text=True)
    if proc.returncode != 0:
        print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
        raise SystemExit("pip install failed — see tail above.")
import transformers, peft, bitsandbytes
print(f"transformers {transformers.__version__} · peft {peft.__version__} · bitsandbytes {bitsandbytes.__version__}")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check + training config
import subprocess, torch
subprocess.run(["nvidia-smi"], check=False)
print()

MODE   = "trial"                 # "trial" first; "full" only after the trial verdict is READY
EXPORT = "messages_2026-10-31"

from train.qlora import train_config
VRAM = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0.0
if VRAM < 20:
    raise SystemExit(f"GPU has {VRAM} GB — pick the L4 (or A100) runtime.")
cfg = train_config(MODE, VRAM)
ADAPTER_REPO = f"{HF_NS}/mbg-qwen7b-lora-{MODE}-{EXPORT}"
EVAL_REPO    = f"{HF_NS}/mbg-eval-{EXPORT}"
for k, v in cfg.items():
    print(f"  {k}: {v}")
print(f"  adapter repo: {ADAPTER_REPO}\n  eval repo:    {EVAL_REPO}")

## Configuration

- Data: private dataset `ankit-iiitdmj/mbg-sft-<EXPORT>` → `train.jsonl`, `validation.jsonl` (200 rows used for
  the validation loss), `golden.jsonl` (the exam)
- Base: `Qwen/Qwen2.5-7B-Instruct` (Apache-2.0), 4-bit NF4, bf16 compute
- LoRA r=16, alpha=32, dropout 0.05 · lr 2e-4 cosine · effective batch 16 · max 1,536 tokens · paged 8-bit AdamW
- Outputs: adapter + checkpoints → `ankit-iiitdmj/mbg-qwen7b-lora-<MODE>-<EXPORT>` (private model repo);
  exam replies + scores → `ankit-iiitdmj/mbg-eval-<EXPORT>` (private dataset)

In [ ]:
# Cell 6 — load the training data + the golden exam from the private HF dataset
import json, random
from collections import Counter
from huggingface_hub import hf_hub_download

def _load(name):
    p = hf_hub_download(f"{HF_NS}/mbg-sft-{EXPORT}", name, repo_type="dataset")
    with open(p, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

train_rows, val_rows, golden = _load("train.jsonl"), _load("validation.jsonl"), _load("golden.jsonl")
rng = random.Random(cfg["seed"])
rng.shuffle(train_rows); rng.shuffle(val_rows)
if cfg["train_rows"]:
    train_rows = train_rows[:cfg["train_rows"]]
val_rows = val_rows[:cfg["eval_rows"]]

print(f"train {len(train_rows):,} · validation {len(val_rows)} · golden {len(golden)}")
print("train by language:", dict(Counter(r["lang"] for r in train_rows)))
print("train hand-over share:", round(sum(r["kind"] == "handover" for r in train_rows) / len(train_rows), 3))
print("golden:", dict(Counter(r["kind"] for r in golden)), dict(Counter(r["lang"] for r in golden)))

In [ ]:
# Cell 7 — tokenise (loss on the reply only) + show exactly what the model learns from one example
from train.qlora import load_tokenizer, tokenize_example

tok = load_tokenizer(cfg["base_model"])
train_ds = [x for x in (tokenize_example(r, tok, cfg["max_len"]) for r in train_rows) if x]
val_ds   = [x for x in (tokenize_example(r, tok, cfg["max_len"]) for r in val_rows) if x]
lens = sorted(len(x["input_ids"]) for x in train_ds)
print(f"train examples {len(train_ds):,} (dropped {len(train_rows) - len(train_ds)} longer than {cfg['max_len']})")
print(f"tokens per example: mean {sum(lens)/len(lens):.0f} · p95 {lens[int(len(lens)*.95)]} · max {lens[-1]}"
      f" · total {sum(lens)/1e6:.2f}M per epoch")

ex = train_ds[0]
learned = tok.decode([t for t in ex["labels"] if t != -100])
print("\nPROMPT TAIL :", repr(tok.decode(ex["input_ids"][:len(ex["input_ids"]) - len([t for t in ex["labels"] if t != -100])])[-160:]))
print("LEARNED PART:", repr(learned))
assert learned.rstrip().endswith("<|im_end|>") and "<|im_start|>" not in learned, "masking is wrong — stop here"
print("masking ok")

In [ ]:
# Cell 8 — repos + resume state (what earlier sessions already finished)
from huggingface_hub import HfApi
from train.qlora import adapter_finished, hub_checkpoint_steps

api = HfApi()
api.create_repo(ADAPTER_REPO, private=True, exist_ok=True)
api.create_repo(EVAL_REPO, repo_type="dataset", private=True, exist_ok=True)
FINISHED = adapter_finished(ADAPTER_REPO)
eval_files = set(api.list_repo_files(EVAL_REPO, repo_type="dataset"))
print(f"adapter repo {ADAPTER_REPO}: private={api.repo_info(ADAPTER_REPO).private}")
print(f"  training finished earlier: {FINISHED} · checkpoints on the Hub: {hub_checkpoint_steps(ADAPTER_REPO)}")
print(f"eval repo {EVAL_REPO}: {sorted(eval_files)}")

In [ ]:
# Cell 9 — load the base model in 4-bit + fresh LoRA adapters (fresh adapters = still the untrained model)
from train.qlora import load_model
model = load_model(cfg)
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# Cell 10 — the exam helper + BASELINE exam (untrained model; cached on HF after the first run)
import os, tempfile
from mbg_eval.metrics import score, summarize, by_group
from mbg_eval.generate import generate_replies

def run_exam(name):
    """Golden replies + scores for `name`; downloaded if an earlier session already ran it."""
    f_preds, f_sum = f"{name}_preds.jsonl", f"{name}_summary.json"
    if f_preds in eval_files:
        p = hf_hub_download(EVAL_REPO, f_preds, repo_type="dataset")
        with open(p, encoding="utf-8") as fh:
            by_id = {r["item_id"]: r["pred"] for r in map(json.loads, fh)}
        preds = [by_id[r["item_id"]] for r in golden]
        print(f"{name}: loaded {len(preds)} cached replies from the Hub")
    else:
        print(f"{name}: generating {len(golden)} replies ...")
        preds = generate_replies(model, tok, golden)
    scores = [score(r, p) for r, p in zip(golden, preds)]
    summary = summarize(scores)
    summary["by_language"] = by_group(golden, scores, "lang")
    summary["by_kind"] = by_group(golden, scores, "kind")
    if f_preds not in eval_files:
        tmp = tempfile.mkdtemp()
        with open(os.path.join(tmp, f_preds), "w", encoding="utf-8") as fh:
            for r, p, s in zip(golden, preds, scores):
                fh.write(json.dumps({"item_id": r["item_id"], "pred": p, **{k: s[k] for k in
                         ("passed", "problems", "expected_handover", "pred_handover")}}, ensure_ascii=False) + "\n")
        with open(os.path.join(tmp, f_sum), "w", encoding="utf-8") as fh:
            json.dump(summary, fh, indent=2, ensure_ascii=False)
        api.upload_folder(folder_path=tmp, repo_id=EVAL_REPO, repo_type="dataset", commit_message=f"exam: {name}")
        eval_files.update({f_preds, f_sum})
    print(json.dumps({k: v for k, v in summary.items() if k not in ("by_language", "by_kind")}, indent=1))
    return preds, summary

BASE_NAME = "baseline_qwen2.5-7b"
base_preds, base_sum = run_exam(BASE_NAME)

## Training

Resumes from the last checkpoint on the Hub if an earlier session was cut off; skips straight to the trained
adapter if training already finished. The validation loss is measured before training and at every eval step —
it must go down.

In [ ]:
# Cell 11 — QLoRA training (resumable)
import time, math
from transformers import Trainer
from train.qlora import Collator, training_args, make_callbacks, download_latest_checkpoint, load_trained_adapter

if FINISHED:
    load_trained_adapter(model, ADAPTER_REPO)
else:
    args = training_args(cfg, "/content/outputs", len(train_ds))
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=Collator(tok.pad_token_id), callbacks=make_callbacks(ADAPTER_REPO))
    resume = download_latest_checkpoint(ADAPTER_REPO, "/content/hub_ckpt")
    if "eval_before.json" in api.list_repo_files(ADAPTER_REPO):
        with open(hf_hub_download(ADAPTER_REPO, "eval_before.json")) as fh:
            loss_before = json.load(fh)["eval_loss"]
    else:
        loss_before = trainer.evaluate()["eval_loss"]
        with open("/content/eval_before.json", "w") as fh:
            json.dump({"eval_loss": loss_before}, fh)
        api.upload_file(path_or_fileobj="/content/eval_before.json", path_in_repo="eval_before.json",
                        repo_id=ADAPTER_REPO, commit_message="validation loss before training")
    print(f"validation loss before training: {loss_before:.4f}")
    print("resuming from", resume) if resume else print("starting fresh")

    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume)
    minutes = (time.time() - t0) / 60
    losses = [float(h["loss"]) for h in trainer.state.log_history if "loss" in h]
    if any(math.isnan(l) or l > 10 for l in losses):
        raise SystemExit("ABORT: training loss broke (NaN / exploded) — send this output to Claude.")
    loss_after = trainer.evaluate()["eval_loss"]
    if losses:
        print(f"\ntrain loss first/last logged: {losses[0]:.3f} -> {losses[-1]:.3f}")
    print(f"validation loss: {loss_before:.4f} -> {loss_after:.4f}")
    if loss_after >= loss_before:
        raise SystemExit("ABORT: validation loss did not go down — send this output to Claude.")

    model.save_pretrained("/content/final_adapter")
    with open("/content/final_adapter/train_summary.json", "w") as fh:
        json.dump({"mode": MODE, "export": EXPORT, "config": cfg, "train_examples": len(train_ds),
                   "minutes_this_session": round(minutes, 1), "train_losses": losses,
                   "eval_loss_before": loss_before, "eval_loss_after": loss_after,
                   "log_history": trainer.state.log_history}, fh, indent=1)
    api.upload_folder(folder_path="/content/final_adapter", repo_id=ADAPTER_REPO,
                      commit_message=f"final adapter ({MODE})")
    print(f"trained in {minutes:.0f} min this session · final adapter pushed -> {ADAPTER_REPO}")

In [ ]:
# Cell 12 — the same exam with the trained adapter
ft_preds, ft_sum = run_exam(f"{MODE}_qlora")

## Comparison + verdict

**READY** needs all of: pass rate at least 10 points above the baseline, hand-over decision right ≥ 75%,
hand-over format right ≥ 90%, facts grounded ≥ 95% of answers.

In [ ]:
# Cell 13 — comparison table, examples side by side, verdict
from mbg_eval.metrics import compare_table

print(compare_table({"baseline": base_sum, MODE: ft_sum}))
print("\npass rate by language:")
for lang in sorted(ft_sum["by_language"]):
    print(f"  {lang:<9} baseline {base_sum['by_language'][lang]['pass_rate']:.2f} -> {MODE} "
          f"{ft_sum['by_language'][lang]['pass_rate']:.2f}  (n={ft_sum['by_language'][lang]['n']})")
print("\nproblems baseline:", base_sum["problems"])
print(f"problems {MODE}:   ", ft_sum["problems"])

random.seed(1)
for i in random.sample(range(len(golden)), 8):
    r = golden[i]
    print(f"\n[{r['lang']} · {r['topic']} · expected {'HAND-OVER' if r['reference'].startswith('[HANDOVER]') else 'answer'}]")
    print(f"  CUSTOMER : {r['messages'][-1]['content'][:160]!r}")
    print(f"  BASELINE : {base_preds[i][:220]!r}")
    print(f"  {MODE.upper():<9}: {ft_preds[i][:220]!r}")

ok = (ft_sum["pass_rate"] >= base_sum["pass_rate"] + 0.10 and ft_sum["decision_accuracy"] >= 0.75
      and ft_sum["handover_format_ok"] >= 0.90 and ft_sum["grounded_rate"] >= 0.95)
nxt = "run the FULL training (Cell 4: MODE = 'full')" if MODE == "trial" else "Level 4 review by Claude"
print("\nVERDICT:", f"READY — {nxt}" if ok else "CHECK — send this output to Claude")

## Decision tree

- **READY (trial)** — send the Cell 13 output to Claude, then set `MODE = "full"` in Cell 4 and run all again
  (the baseline exam is cached, so it is skipped).
- **READY (full)** — send the output to Claude: Level 4 review of the exam replies, then packaging.
- **CHECK / ABORT** — send the output to Claude. Nothing is lost: checkpoints and exam replies are on the Hub.
- **Session cut off** — run all again: training resumes from the last checkpoint on the Hub.